# Ollama LLM-as-Judge Evaluation

Goal: use a local Ollama model to automatically judge RAG outputs.

In the previous notebook, we manually scored retrieval relevance, correctness, and faithfulness. In this notebook, we will ask a local LLM to produce those scores for us.


## Why This Matters

Manual eval is great for learning, but it does not scale. If we have 100 questions and 5 RAG configurations, manual scoring becomes slow and inconsistent.

LLM-as-judge lets us run repeatable evaluation loops:

```text
question + expected answer + retrieved context + generated answer
→ judge prompt
→ local LLM score
→ metrics we can compare
```

This is not perfect, but it is a useful bridge toward production evals.


## Agenda

1. Rebuild the local baseline RAG pipeline.
2. Create a small eval dataset.
3. Run RAG and collect generated answers + retrieved context.
4. Write local judge prompts for:
   - retrieval relevance
   - correctness
   - faithfulness
5. Ask Ollama/Gemma to return structured scores.
6. Parse scores into Python dictionaries.
7. Compute average metrics.
8. Compare local judge scores with our manual intuition.


## Score Definitions

We will use the same 0-2 scale from manual eval.

```text
0 = bad / unsupported / irrelevant
1 = partial / weak / incomplete
2 = good / clearly supported / relevant
```

Metrics:

```text
retrieval_relevance
→ do the retrieved chunks contain information needed to answer the question?

correctness
→ does the generated answer match the expected answer?

faithfulness
→ is the generated answer supported by the retrieved context?
```


## Retrieval Relevance vs Faithfulness

Retrieval relevance judges the retrieved context: did the retriever bring useful evidence for the question?

Faithfulness judges the generated answer: did the model's answer stay supported by the retrieved context?

Example:

```text
Question: What causes recent climate change?
Retrieved context: Greenhouse gases such as CO2 and methane trap heat in the atmosphere.
Generated answer: Recent climate change is mainly caused by volcanic eruptions.

Retrieval relevance: high, because the context contains the needed evidence.
Faithfulness: low, because the generated answer is not supported by that context.
```

Short version:

```text
Retrieval relevance = was the evidence good?
Faithfulness = did the answer stick to the evidence?
```


## Important Limitation

A local LLM judge can be wrong. Treat its scores as a signal, not absolute truth. For serious systems, we compare judge outputs against human-reviewed examples and keep improving the judge prompt/eval dataset.


In [9]:
from pathlib import Path
import json
import re

import ollama

from langchain_huggingface import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import TextSplitter

MODEL_NAME = "gemma4:e4b"


In [ ]:
# Now we are gonna be building baseline RAG

pdf_path = Path("../data/Understanding_Climate_Change.pdf")

loader = PyPDFLoader(pdf_path)
documents = loader.load()

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 1000,
    chunk_overlap = 200,
    length_function= len
)

# print([ele for ele in dir(RecursiveCharacterTextSplitter) if not ele.startswith("__")])

chunks = text_splitter.split_documents(documents)
print(len(chunks))


97


In [14]:
# initalize embedding model

embedding_model = HuggingFaceEmbeddings(
    model="sentence-transformers/all-MiniLM-L6-v2"
)

# print(help(HuggingFaceEmbeddings))

In [18]:
vector_store = FAISS.from_documents(chunks, embedding_model)

print(type(vector_store))
print(help(vector_store))

<class 'langchain_community.vectorstores.faiss.FAISS'>
Help on FAISS in module langchain_community.vectorstores.faiss object:

class FAISS(langchain_core.vectorstores.base.VectorStore)
 |  FAISS(embedding_function: 'Union[Callable[[str], List[float]], Embeddings]', index: 'Any', docstore: 'Docstore', index_to_docstore_id: 'Dict[int, str]', relevance_score_fn: 'Optional[Callable[[float], float]]' = None, normalize_L2: 'bool' = False, distance_strategy: 'DistanceStrategy' = <DistanceStrategy.EUCLIDEAN_DISTANCE: 'EUCLIDEAN_DISTANCE'>)
 |  
 |  FAISS vector store integration.
 |  
 |  See [The FAISS Library](https://arxiv.org/pdf/2401.08281) paper.
 |  
 |  Setup:
 |      Install ``langchain_community`` and ``faiss-cpu`` python packages.
 |  
 |      .. code-block:: bash
 |  
 |          pip install -qU langchain_community faiss-cpu
 |  
 |  Key init args — indexing params:
 |      embedding_function: Embeddings
 |          Embedding function to use.
 |  
 |  Key init args — client params:

In [19]:
print("PDF exists:", pdf_path.exists())
print("Page documents:", len(documents))
print("Chunks:", len(chunks))
print("Vector store:", type(vector_store))

PDF exists: True
Page documents: 33
Chunks: 97
Vector store: <class 'langchain_community.vectorstores.faiss.FAISS'>


# EVALS

In [25]:
eval_dataset = [
    {
        "question": "What is the main cause of recent climate change?",
        "expected_answer": "The increase in greenhouse gases in the atmosphere, mainly from human activities such as burning fossil fuels and deforestation.",
    },
    {
        "question": "How does burning fossil fuels contribute to climate change?",
        "expected_answer": "Burning fossil fuels releases large amounts of carbon dioxide, increasing greenhouse gases and warming the climate.",
    },
    {
        "question": "What are some effects of climate change?",
        "expected_answer": "Effects include rising temperatures, melting ice, sea-level rise, extreme weather, droughts, flooding, and ocean acidification.",
    },
    {
        "question": "How does renewable energy help mitigate climate change?",
        "expected_answer": "Renewable energy sources produce little to no greenhouse gas emissions and reduce reliance on fossil fuels.",
    },
    {
        "question": "What is ocean acidification?",
        "expected_answer": "Ocean acidification is the increase in ocean acidity caused by higher CO2 levels dissolving into seawater.",
    },
]

print("Eval questions:", len(eval_dataset))

Eval questions: 5


In [28]:

def run_rag(question, k=3):
    retrieved_docs = vector_store.similarity_search(question, k)
    context = "\n\n".join(ele.page_content for ele in retrieved_docs)

    prompt = f"""
        Answer the question using only the context below.
        If the answer is not in the context, say: "I don't know based on the context provided"
        Question: {question}
        Context: {context}

        Answer in one short sentence"
    """

    response = ollama.chat(
        model=MODEL_NAME,
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ]
    )

    return {
        "question": question,
        "generated_answer": response.message.content,
        "context": context,
        "retrieved_docs": retrieved_docs
    }


In [ ]:
# TESTING

sample = run_rag(eval_dataset[0]["question"], k=3)

print("Question")
print(sample["question"])

print("\nGenerated answer:", sample["generated_answer"])
print("\nContext Preview:-")
print(sample["context"][:800])

Question
What is the main cause of recent climate change?

 Generated answer: The primary cause of recent climate change is the increase in greenhouse gases in the atmosphere.

Context Preview:-
Most of these climate changes are attributed to very small variations in Earth's orbit that 
change the amount of solar energy our planet receives. During the Holocene epoch, which 
began at the end of the last ice age, human societies flourished, but the industrial era has seen 
unprecedented changes. 
Modern Observations 
Modern scientific observations indicate a rapid increase in global temperatures, sea levels, 
and extreme weather events. The Intergovernmental Panel on Climate Change (IPCC) has 
documented these changes extensively. Ice core samples, tree rings, and ocean sediments 
provide a historical record that scientists use to understand past climate conditions and 
predict future trends. The evidence overwhelmingly shows that recent changes are primarily 
driven by human activi


Now we will create the first judge function::: 

question+ expected_answer+ generated_answer
-> score - 0/1/2

In [ ]:
# ollama may return extra text around JSON.
# This helper extracts the JSON obj from the resonse

def extract_json(text):
    match = re.search(r"\{.*\}", text, re.DOTALL)

    if not match: 
        raise ValueError(f"No JSON object found in response: \n{text}")
    
    return json.loads(match.group(0))


temp = {
    "name": "jatin",
    "age" : 25
}

json_str = json.dumps(temp)
print(extract_json(json_str))

{'name': 'jatin', 'age': 25}


# Local judge for correctness

In [52]:

def judge_correctness(question, expected_answer, generated_answer):
    prompt = f"""
        you are evaluating a RAG system answer.
        Score correctness on a 0-2 scale:
        0 = generated answer is wrong or does not answer the question
        1 = generated answer is partially correct but incomplete
        2 = generated answer correctly answers the question

        Question: {question}
        Expected answer: {expected_answer}
        Generated answer: {generated_answer}

        Return valid JSON in this exact form:
        {{
            "score": 0,
            "reason": "short explanation"
        }}
    """
    response = ollama.chat(
        model=MODEL_NAME,
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ]
    )

    return extract_json(response.message.content)

In [ ]:
result = judge_correctness(
    question=eval_dataset[0]["question"],
    expected_answer=eval_dataset[0]["expected_answer"],
    generated_answer=sample["generated_answer"] 
    )

print(result)

The generated answer correctly identifies the main cause (increase in greenhouse gases) but is incomplete because it fails to mention the key sources of these gases, such as human activities like burning fossil fuels or deforestation, which were detailed in the expected answer.


Correctness asks -- Does the answer match the expected answer?
Faithfulness asks -- Is the answer supported by the retrieved context?

In [57]:
def judge_faithfulness(question, context, generated_answer):
    prompt = f"""
You are evaluating whether a RAG answer is supported by the retrieved context.

Score faithfulness on a 0-2 scale:
0 = answer is not supported by the context or contradicts it
1 = answer is partially supported by the context
2 = answer is clearly supported by the context

Question:
{question}

Retrieved context:
{context}

Generated answer:
{generated_answer}

Return only valid JSON in this exact format:
{{
  "score": 0,
  "reason": "short explanation"
}}
"""

    response = ollama.chat(
        model=MODEL_NAME,
        messages=[
            {"role": "user", "content": prompt}
        ],
    )

    return extract_json(response.message.content)

Faithfulness catches hallucination.

In [58]:
faithfulness_result = judge_faithfulness(
    question=sample["question"],
    context=sample["context"],
    generated_answer=sample["generated_answer"],
)

print(faithfulness_result)

{'score': 2, 'reason': "The context directly states that 'The primary cause of recent climate change is the increase in greenhouse gases in the atmosphere' under Chapter 2: Causes of Climate Change."}


Retrieval Relevance - Did the retriever bring useful chunks for this question?

In [ ]:
def judge_retrieval_relevance(question, context):
    prompt = f"""
You are evaluating retrieved context for a RAG system.

Score retrieval relevance on a 0-2 scale:
0 = retrieved context is irrelevant and does not help answer the question
1 = retrieved context is partially relevant but incomplete or noisy
2 = retrieved context clearly contains information needed to answer the question

Question:
{question}

Retrieved context:
{context}

Return only valid JSON in this exact format:
{{
  "score": 0,
  "reason": "short explanation"
}}
"""

    response = ollama.chat(
        model=MODEL_NAME,
        messages=[
            {"role": "user", "content": prompt}
        ],
    )

    return extract_json(response.message.content)
    

In [60]:
retrieval_result = judge_retrieval_relevance(
    question=sample["question"],
    context=sample["context"],
)

print(retrieval_result)

{'score': 2, 'reason': "The retrieved context repeatedly and clearly identifies the main cause, stating that 'The primary cause of recent climate change is the increase in greenhouse gases in the atmosphere' and attributing this increase to human activities."}


retrieval_relevance low + correctness low
→ retrieval is likely bad

retrieval_relevance high + correctness low
→ generation/prompt/model is likely bad

faithfulness low
→ hallucination or unsupported answer

This is an automated regression eval loop.

In [61]:
def evaluate_rag_result(item, k=3):
    rag_result = run_rag(item["question"], k=k)

    correctness = judge_correctness(
        question=item["question"],
        expected_answer=item["expected_answer"],
        generated_answer=rag_result["generated_answer"],
    )

    faithfulness = judge_faithfulness(
        question=item["question"],
        context=rag_result["context"],
        generated_answer=rag_result["generated_answer"],
    )

    retrieval_relevance = judge_retrieval_relevance(
        question=item["question"],
        context=rag_result["context"],
    )

    return {
        "question": item["question"],
        "expected_answer": item["expected_answer"],
        "generated_answer": rag_result["generated_answer"],
        "context": rag_result["context"],
        "correctness": correctness,
        "faithfulness": faithfulness,
        "retrieval_relevance": retrieval_relevance,
    }

In [62]:
judge_results = []

for i, item in enumerate(eval_dataset, start=1):
    print(f"Evaluating {i}/{len(eval_dataset)}: {item['question']}")
    result = evaluate_rag_result(item, k=3)
    judge_results.append(result)

print("Done:", len(judge_results))

Evaluating 1/5: What is the main cause of recent climate change?
Evaluating 2/5: How does burning fossil fuels contribute to climate change?
Evaluating 3/5: What are some effects of climate change?
Evaluating 4/5: How does renewable energy help mitigate climate change?
Evaluating 5/5: What is ocean acidification?
Done: 5


In [63]:
def average_judge_score(results, metric_name):
    scores = []

    for result in results:
        score = result[metric_name]["score"]
        scores.append(score)

    return sum(scores) / len(scores)


avg_correctness = average_judge_score(judge_results, "correctness")
avg_faithfulness = average_judge_score(judge_results, "faithfulness")
avg_retrieval_relevance = average_judge_score(judge_results, "retrieval_relevance")

print("Average correctness:", avg_correctness, "=>", (avg_correctness / 2) * 100, "%")
print("Average faithfulness:", avg_faithfulness, "=>", (avg_faithfulness / 2) * 100, "%")
print("Average retrieval relevance:", avg_retrieval_relevance, "=>", (avg_retrieval_relevance / 2) * 100, "%")

Average correctness: 1.4 => 70.0 %
Average faithfulness: 2.0 => 100.0 %
Average retrieval relevance: 2.0 => 100.0 %
